# Lekcija 36: Klasifikacija slika u praksi

Svaki dosadašnji klasifikator evaluiran je jednim brojem: ukupnom točnošću. Taj broj mnogo toga skriva. Ova lekcija gradi klasifikator s 4 klase na namjerno neuravnoteženom podskupu **CIFAR-10** — stvarnim fotografijama umjesto sintetičkih oblika — i pokazuje zašto sama točnost može zavarati, koristeći alate koji otkrivaju što zapravo ne radi: **matricu zabune** te **preciznost i odziv po klasama**.

In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Neuravnoteženi skup podataka s 4 klase

Četiri kategorije stvarnih fotografija iz **CIFAR-10** (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009.</a>): mačka, pas, automobil i kamion. Testni skup ima jednakih 150 slika po klasi, ali skup za *treniranje* namjerno sadrži malo kamiona (90 primjera, nasuprot 400 za svaku od ostalih triju klasa) — kao prikaz čestog stvarnog slučaja u kojem je neke klase teže prikupiti od drugih. Mačka i pas namjerno su uključeni kao vizualno sličan par, neovisno o količini podataka za treniranje svake klase.

In [ ]:
CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

ensure_cifar10()

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
CLASSES = ['cat', 'dog', 'automobile', 'truck']
CLASS_IDS = {name: CIFAR_LABELS.index(name) for name in CLASSES}

train_imgs, train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    train_imgs.append(imgs); train_labels.append(labels)
train_imgs, train_labels = np.concatenate(train_imgs), np.concatenate(train_labels)
test_imgs, test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

rng = np.random.default_rng(7)

def subset_for_classes(imgs, labels, per_class_counts):
    out_imgs, out_labels = [], []
    for i, name in enumerate(CLASSES):
        idx = np.where(labels == CLASS_IDS[name])[0]
        idx = rng.permutation(idx)[:per_class_counts[name]]
        out_imgs.append(imgs[idx])
        out_labels.append(np.full(len(idx), i, dtype=np.int64))
    out_imgs, out_labels = np.concatenate(out_imgs), np.concatenate(out_labels)
    perm = rng.permutation(len(out_imgs))
    return out_imgs[perm], out_labels[perm]

# training set: cat/dog/automobile well represented, truck deliberately starved
X_train, y_train = subset_for_classes(train_imgs, train_labels,
    {'cat': 400, 'dog': 400, 'automobile': 400, 'truck': 90})
# test set: balanced across all four, so the imbalance is purely a training-time artifact
X_test, y_test = subset_for_classes(test_imgs, test_labels,
    {'cat': 150, 'dog': 150, 'automobile': 150, 'truck': 150})

print('training class counts:', dict(zip(CLASSES, np.bincount(y_train, minlength=4))))
print('test class counts:    ', dict(zip(CLASSES, np.bincount(y_test, minlength=4))))

fig, axes = plt.subplots(1, 4, figsize=(8, 2.2))
for ax, name in zip(axes, CLASSES):
    ax.imshow(X_train[y_train == CLASSES.index(name)][0])
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.show()

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )
        self.fc = nn.Linear(64, 4)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))

torch.manual_seed(0)
model = CNN()
opt = torch.optim.Adam(model.parameters(), lr=0.0007)
Xt = torch.tensor(X_train).permute(0, 3, 1, 2); yt = torch.tensor(y_train)
for _ in range(150):
    opt.zero_grad()
    loss = F.cross_entropy(model(Xt), yt)
    loss.backward()
    opt.step()

Xte = torch.tensor(X_test).permute(0, 3, 1, 2)
with torch.no_grad():
    preds = model(Xte).argmax(1).numpy()

acc = (preds == y_test).mean()
print(f'overall test accuracy: {acc:.1%}')

Ukupna točnost malo iznad 50 % na zadatku s 4 klase, gdje slučajno pogađanje daje 25 %, zvuči razumno za mali CNN treniran od početka na stvarnim fotografijama. No skriva važnu činjenicu: model nije jednako dobar za sve četiri klase.

## Matrica zabune

Redak `i`, stupac `j` broji testne primjere stvarne klase `i` predviđene kao klasa `j`. Savršen klasifikator daje dijagonalnu matricu; sve izvan dijagonale konkretna je pogreška koju možemo imenovati.

In [ ]:
cm = np.zeros((4, 4), dtype=int)
for t, p in zip(y_test, preds):
    cm[t, p] += 1

print('confusion matrix (rows=true, cols=predicted):')
print(f'{"":>10}' + ''.join(f'{s:>10}' for s in CLASSES))
for i, s in enumerate(CLASSES):
    print(f'{s:>10}' + ''.join(f'{cm[i, j]:>10}' for j in range(4)))

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES, rotation=45)
ax.set_yticks(range(4)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.title('Confusion matrix')
plt.tight_layout()
plt.show()

## Stvarno pozitivni, lažno pozitivni, stvarno negativni i lažno negativni rezultati

Svaka ćelija gornje matrice zabune određena je vrsta točnosti ili pogreške, ali standardni rječnik kojim ih opisujemo binaran je: odaberite jednu klasu i pitajte samo „pripada li joj ili ne?” Svođenje matrice s 4 klase na „kamion nasuprot svemu ostalom” daje točno četiri ishoda:

- **Stvarno pozitivan (TP)**: doista kamion, predviđen kamion. Pogodak.
- **Lažno negativan (FN)**: doista kamion, predviđeno nešto drugo. Promašaj — model ga je propustio.
- **Lažno pozitivan (FP)**: zapravo *nije* kamion, ali ipak je predviđen kao kamion. Lažna uzbuna.
- **Stvarno negativan (TN)**: zapravo nije kamion, ispravno predviđen kao nešto što nije kamion.

To je ista tablica 2x2 na kojoj počiva evaluacija svakog binarnog klasifikatora („pozitivan/negativan” rezultat medicinskog testa, odluka filtra neželjene pošte „neželjena/nije neželjena”) — višeklasna matrica zabune samo je ta tablica izračunata za svaku klasu, pri čemu se sve izvan retka i stupca te klase svodi na „nije ta klasa”.

In [ ]:
cls = CLASSES.index('truck')
tp = cm[cls, cls]
fn = cm[cls, :].sum() - tp          # true truck, predicted something else
fp = cm[:, cls].sum() - tp          # predicted truck, actually something else
tn = cm.sum() - tp - fn - fp        # everything else, correctly not called truck

print(f'{"":>18}{"predicted truck":>20}{"predicted NOT truck":>24}')
print(f'{"actually truck":>18}{tp:>20}{fn:>24}')
print(f'{"actually NOT truck":>18}{fp:>20}{tn:>24}')
print()
print(f'TP={tp}, FP={fp}, FN={fn}, TN={tn}, total={tp+fp+fn+tn} (test set size={len(y_test)})')

## Preciznost i odziv

Dva broja po klasi, izračunata izravno iz TP/FP/FN:
- **Odziv** = TP / (TP + FN) — od svega što je stvarno *pripadalo* klasi `i`, koji je udio model otkrio? Nizak odziv znači da model često propušta tu klasu.
- **Preciznost** = TP / (TP + FP) — od svega što je model *nazvao* klasom `i`, koji je udio stvarno pripadao toj klasi? Niska preciznost znači da model za tu klasu često daje lažne uzbune.

(Rjeđe potreban, ali povezan par, izgrađen iz ostalih dvaju kvadranata: **specifičnost** = TN / (TN + FP), koliko dobro model izbjegava lažne uzbune na negativnoj klasi, i njezin komplement, **stopa lažno pozitivnih rezultata** = FP / (FP + TN) = 1 − specifičnost.)

In [ ]:
print(f'{"class":>10} {"precision":>10} {"recall":>8} {"support":>8}')
for i, name in enumerate(CLASSES):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    recall = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    support = cm[i, :].sum()
    print(f'{name:>10} {precision:>10.2f} {recall:>8.2f} {support:>8}')

Kamion — klasa ograničena na 90 slika za treniranje, nasuprot 400 za svaku drugu — ima visoku preciznost, ali vrlo nizak odziv: kada model kaže „kamion”, obično je u pravu, ali ne prepoznaje golemu većinu stvarnih kamiona. Umjesto toga bira klase kojih je pri treniranju vidio mnogo (uglavnom „automobil”, vizualno najbližu dobro zastupljenu klasu). To je tipičan znak neuravnoteženosti klasa i potpuno je nevidljiv u prethodnom jednom broju ukupne točnosti. Nasuprot tome, mačka i pas dobro su zastupljeni pri treniranju, ali model ih međusobno zamjenjuje mnogo češće nego s automobilom ili kamionom — potpuno drukčiji način neuspjeha, uzrokovan stvarnom vizualnom sličnošću dviju životinja umjesto nedostatkom podataka. Automobil završava s neuobičajeno visokim odzivom, ali samo osrednjom preciznošću, iz istog razloga zbog kojeg je stradao odziv kamiona: prima pogrešne klasifikacije slabo zastupljene klase kamiona, jer je „vozilo na kotačima” jednostavno rezervno predviđanje kada model ne može razlikovati kamione od automobila.

Praktična pouka: uvijek pregledajte matricu zabune i metrike po klasama prije nego što povjerujete jednom broju točnosti, osobito na skupovima podataka u kojima klase nisu prirodno uravnotežene.

### Zadaci

1. Povećajte broj kamiona za treniranje s 90 na 400 (kao za ostale tri klase) i ponovno pokrenite. Oporavlja li se odziv kamiona? Poboljšava li se i razlikovanje mačaka i pasa ili zabuna ostaje — i zašto više podataka ne bi riješilo problem uzrokovan stvarnom vizualnom sličnošću umjesto oskudicom podataka?
2. Pokušajte ponderirati gubitak inverznom frekvencijom klasa (`F.cross_entropy(logits, yt, weight=class_weights)`, gdje je `class_weights[i] = 1 / count(class i)`) umjesto prikupljanja više slika kamiona. Vraća li to odziv kamiona i uz kakav utjecaj na preciznost ostalih klasa?
3. Izračunajte **F1 mjeru** (harmonijsku sredinu preciznosti i odziva, `2 * p * r / (p + r)`) za svaku klasu. Zašto bi F1 mogao biti bolji pojedinačni broj za praćenje po klasi od točnosti, kada točnost ima smisla samo ukupno?